In [17]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))


In [18]:

from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate, SystemMessagePromptTemplate, MessagesPlaceholder
, ChatPromptTemplate
from langchain_openrouter import ChatOpenRouter
from pydantic import BaseModel, Field
from src.code_civil_langchain.openrouter_embedding import CustomEmbedOpenRouter
from typing import List
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import chromadb
import os

In [19]:
load_dotenv()

True

In [20]:
apikey =os.getenv("OPENROUTER_API_KEY")

In [75]:
system_prompt = SystemMessagePromptTemplate.from_template(""" Tu es un assistant juriste specialie dans le droit civil.

Ton objectif est de repondre aux questions des clients en te basant sur le code civil.

IMPORTANT
Ne fais pas de supposition et n'invente RIEN qui n'est pas ecris dans le code civil.
toutes tes reponses doivent etres sourcees et contenir le ou les articles du code civil auxquels tu fais reference dans ta reponse.

Pour repondre aux questions des clients, il faut que tu reformule leurs questionsdans un langage qui puisse faciliter 
l'obtention des articles pertinents.

""")

In [76]:
prompt_few_shot = PromptTemplate(template="Client : {ex}\nToi : {ans}\n",input_variables=["ex","ans"])


In [81]:
# TO DO : ADD REAL EXAMPLES
ex = [
  { "ex": "What is the capital of France?", "ans": "Paris" },
  { "ex": "What is the capital of Japan?", "ans": "Tokyo" },
]


In [78]:
print(prompt_few_shot.format(**ex[0]))

Client : What is the capital of France?
Toi : Paris



In [84]:
few_shot_prompt = FewShotPromptTemplate(
    examples=ex,
    example_prompt=prompt_few_shot,
    prefix="Voici quelques exemples de raisonnements attendus pour repondre a un client :",
    suffix="Client question : {input}",
    input_variables=["input"],
    example_separator="\n",
)

In [ ]:
formatted_prompt = few_shot_prompt.format(input="En tant que citoyen francais, ai-je le droit de tuer ?")
print(formatted_prompt)

In [89]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful career coach."),
   # MessagesPlaceholder("conversation"), 
    ("human", "{current_question}")
])


In [21]:
model = ChatOpenRouter(model="nvidia/nemotron-3-ultra-550b-a55b:free",api_key=apikey)

In [94]:

chain = chat_prompt | model | StrOutputParser()

In [ ]:
chain.invoke({"current_question":"Hello!"})

In [107]:

client = chromadb.HttpClient(host="localhost", port=8000, ssl=False)

In [109]:
collections = client.list_collections()

In [110]:
collection

[Collection(name=articles_code_civil)]

In [111]:
collection = client.get_collection("articles_code_civil")

In [122]:
similars = collection.query(query_embeddings=[queryEmb],n_results=4)

In [123]:
similars

{'ids': [['7b4e89db-3e4d-4e89-8390-4533988d38c8',
   'b4695e2c-41d8-49b0-93e2-14c9d4f34229',
   'c7fab72f-2472-4e4c-9d50-696adfb437ca',
   '33cd695d-76ba-4716-a746-8fa698330764']],
 'distances': [[1.0344296, 1.0722449, 1.0924016, 1.1053252]],
 'embeddings': None,
 'metadatas': [[{'source': 'Article  2278 du code civil'},
   {'source': 'Article  1966 du code civil'},
   {'source': 'Article  494-5 du code civil'},
   {'source': 'Article  544 du code civil'}]],
 'documents': [["\nLa possession est protégée, sans avoir égard au fond du droit, contre le trouble qui l'affecte ou la menace.\n\nLa protection possessoire est pareillement accordée au détenteur contre tout autre que celui de qui il tient ses droits.\n\n## Livre IV : Des sûretés\n\n",
   "\nLes jeux propres à exercer au fait des armes, les courses à pied ou à cheval, les courses de chariot, le jeu de paume et autres jeux de même nature qui tiennent à l'adresse et à l'exercice du corps, sont exceptés de la disposition précédente.\n

In [ ]:
chain = chat_prompt | model | StrOutputParser() | 

In [115]:
customEmb = CustomEmbedOpenRouter()

In [117]:
queryEmb = customEmb.embed_query("Ais-je le droit de detenir des armes ?")

In [128]:
class ArticlesQueries(BaseModel):
    query1 : str = Field(description="Rephrasing of client question to query the vector database")
    query2 : str = Field(description="Second rephrasing of client question")
    query3 : str = Field(description="Third rephrasing of client question")
    keywords : List[str] = Field(description="List of keywords relevant to the client question to find better matches in the vector database")
    

In [133]:
structured_model = model.with_structured_output(ArticlesQueries)

In [134]:
structured_model.invoke("\n\nClient question :\n\nAis-je le droit de porter une arme en france ?")

ArticlesQueries(query1="droit de porter une arme en France réglementation port d'arme", query2="législation française sur le port d'arme conditions autorisation", query3='armes autorisées port public France loi armes', keywords=["port d'arme", 'France', 'législation', 'autorisation', 'réglementation', 'arme à feu', 'arme blanche'])